# NephroQ · Renal risk prediction in type 2 diabetes

Notebook ready for Google Colab. Enter a patient's markers
(blood + urine + blood pressure) and get: the projected eGFR trajectory,
the estimated time to dialysis, and a risk classification -- using
NephroQ, the project's calibrated mechanistic model.

**Not a diagnostic tool.** A research prototype (TRL4) to explore with
physicians, not for clinical decisions.

Repository: https://github.com/<your-username>/nephroq

## 1. Set up the environment

In [ ]:
!git clone -q https://github.com/<your-username>/nephroq.git repo 2>/dev/null || echo 'If the repo is private, upload the .py files from src/ to Colab manually instead of cloning.'
%cd repo/src
!pip install -q numpy scipy matplotlib

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from egfr_measurement import egfr_cr, egfr_cr_cys
from mechanistic_twin import MechanisticRenalModel, egfr_of_N, N_of_egfr, DIALYSIS_eGFR

# Calibrated population parameters (from the hierarchical Bayesian fit, see docs/MODEL_DOCUMENTATION.md).
# Update these if you recalibrate with a real cohort (see section 4).
Q_POP   = 1.52      # feedback exponent (hyperfiltration)
KHF_POP = 0.0141     # hyperfiltration strength
W_POP   = np.array([0.0144, 0.0180, 0.0108])  # weights [A1c, UACR, blood pressure]
print('Population parameters loaded: q=%.2f  k_hf=%.4f' % (Q_POP, KHF_POP))

## 2. Enter the patient's markers
Edit this cell with the real values. If you don't have eGFR directly, compute it from creatinine (and cystatin, if available) in the next cell.

In [ ]:
# ---------------- EDIT HERE ----------------
age             = 58        # years
sex             = 'F'       # 'M' or 'F'
creatinine      = 1.3       # mg/dL (blood)
cystatin        = None      # mg/L (blood) -- optional, improves precision; None if not measured
hba1c           = 8.1       # % (blood)
uacr            = 145       # mg/g (urine) -- albumin/creatinine ratio
systolic_bp     = 142       # mmHg
on_sglt2_raas   = False     # True if already on SGLT2i or ACEi/ARB
# ---------------------------------------------

if cystatin:
    egfr0 = egfr_cr_cys(creatinine, cystatin, age, sex)
    print(f'Baseline eGFR (creatinine+cystatin, more precise): {egfr0:.1f} mL/min/1.73m2')
else:
    egfr0 = egfr_cr(creatinine, age, sex)
    print(f'Baseline eGFR (creatinine only): {egfr0:.1f} mL/min/1.73m2')
    print('Suggestion: requesting cystatin C reduces the error in q by ~5x (see docs/MODEL_DOCUMENTATION.md).')

## 3. Trajectory projection and risk

In [ ]:
def gfr_category(egfr):
    """KDIGO GFR category (G3 splits into G3a/G3b)."""
    if egfr >= 90: return 'G1'
    if egfr >= 60: return 'G2'
    if egfr >= 45: return 'G3a'
    if egfr >= 30: return 'G3b'
    if egfr >= 15: return 'G4'
    return 'G5'

def project(a1c, sbp, uacr, egfr0, treated, years=15):
    # Explicit calibrated weights (w_a1c, w_uacr, w_sbp) -> forces N_ref=1, k_met=1
    # internally, matching EXACTLY the parameterization used by calibrate_mimic.py.
    # (No monkeypatching of metabolic_insult() -- avoids silent double-scaling.)
    m = MechanisticRenalModel(a1c=a1c, sbp=sbp, uacr=uacr, u=1.0 if treated else 0.0,
                              k_hf=KHF_POP, q=Q_POP,
                              w_a1c=W_POP[0], w_uacr=W_POP[1], w_sbp=W_POP[2])
    t, N, egfr, t_dial = m.simulate(N_of_egfr(egfr0), years=years)
    return t, egfr, t_dial

t_current, egfr_current, t_dial_current = project(hba1c, systolic_bp, uacr, egfr0, on_sglt2_raas)
t_alt, egfr_alt, t_dial_alt = project(hba1c, systolic_bp, uacr, egfr0, not on_sglt2_raas)

fig, ax = plt.subplots(figsize=(8,5))
label_current = 'Current treatment' if on_sglt2_raas else 'No treatment (current scenario)'
label_alt = 'Illustrative renoprotective scenario added' if not on_sglt2_raas else 'If treatment is stopped'
ax.plot(t_current, egfr_current, lw=2.5, color='#E24B4A', label=label_current)
ax.plot(t_alt, egfr_alt, lw=2.5, color='#1D9E75', label=label_alt)
ax.axhline(DIALYSIS_eGFR, color='k', lw=1, ls='--')
ax.text(0.3, DIALYSIS_eGFR+2, 'modeled eGFR<15 threshold', fontsize=9)
ax.set_xlabel('years'); ax.set_ylabel('projected eGFR (mL/min/1.73m2)')
ax.set_title('Illustrative model projection of renal function'); ax.legend()
plt.show()

print(f"\n{'='*55}")
print(f"Baseline eGFR:                    {egfr0:.1f} mL/min/1.73m2")
print(f"KDIGO GFR category:               {gfr_category(egfr0)}")
print(f"Modeled time to eGFR<15 ({label_current}): "
     f"{t_dial_current:.1f} years" if np.isfinite(t_dial_current) else f"Modeled time to eGFR<15 ({label_current}): >15 years")
print(f"Modeled time to eGFR<15 ({label_alt}): "
     f"{t_dial_alt:.1f} years" if np.isfinite(t_dial_alt) else f"Modeled time to eGFR<15 ({label_alt}): >15 years")
if np.isfinite(t_dial_current) and np.isfinite(t_dial_alt):
    print(f"Estimated difference: {abs(t_dial_alt - t_dial_current):.1f} years")
print('Note: this is a modeled eGFR<15 threshold, not a prediction of dialysis initiation.')
print(f"{'='*55}")


## 4. (Optional) Recalibrate with a real cohort
If you have a CSV with real trajectories (`patient_id, time_years, egfr, hba1c, uacr, sbp`),
run the repo's hierarchical/Bayesian calibration and update `Q_POP`, `KHF_POP`, `W_POP` above with the recalibrated values.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # upload your real CSV here
# import os
# os.environ['CKD_CSV'] = list(uploaded.keys())[0]
# !python mvp_calibration.py
print('Uncomment the lines above to calibrate with your own CSV.')

---
**Notice:** research tool (TRL4), not clinically validated.
Must not be used for medical decisions without qualified supervision.